# Case study: Sachs et al. (2005) flow cytometry data

In [ ]:
import numpy as np
import pandas as pd
from joblib import Parallel, delayed

import case_utils as cu

DATA = "data"
FIGURES = "../figures"
N_JOBS = -1

## Data

In [ ]:
stimuli = cu.load_perturbations(DATA)
pd.Series({k: len(v) for k, v in stimuli.items()}, name="n_cells")

## Preclustering within each perturbation

In [ ]:
summary, silhouette_curves = cu.select_preclusters(stimuli, pca_var=0.8, k_range=range(2, 11), n_init=50, rel_tol=0.01)
summary

In [ ]:
cells = cu.assign_preclusters(stimuli, summary)
sizes = cells.groupby(["stimulus", "cluster"]).size()
kept = cells.set_index(["stimulus", "cluster"]).loc[lambda d: d.index.isin(sizes[sizes >= 20].index)].reset_index()

X_sub = {key: cu.zscore_nan(g[cu.FEATURES].to_numpy()) for key, g in kept.groupby(["stimulus", "cluster"])}
X_input = list(X_sub.values())
print(f"{len(X_input)} subpopulations, {sum(len(X) for X in X_input)} cells")

## Pooled population model

In [ ]:
ALPHA_GRID = [0.001, 0.005, 0.01, 0.1]
alpha_pop, W_pop, cv_pop = cu.tune_notears(np.vstack(X_input), ALPHA_GRID, n_splits=3, seed=42)
cv_pop

## Per-perturbation models

In [ ]:
X_pert = [cu.to_scaled_X(stimuli[f"stimulus{k}"], cu.FEATURES) for k in range(1, 10)]
res_pert = Parallel(n_jobs=N_JOBS)(delayed(cu.tune_notears)(X, ALPHA_GRID, 3, 42) for X in X_pert)
cv_pert = np.average([min(r[2].values()) for r in res_pert], weights=[len(X) for X in X_pert])

## DAG-DC-ADMM

The full grid search is slow (several hours on one core). Set `RUN_CV = False` to skip it.

In [ ]:
RUN_CV = True
PARAM_GRID = {
    "lambda1": [0.001, 0.005, 0.01, 0.1],
    "lambda2": [0.1, 0.05, 0.03, 0.01, 0.005, 0.001, 0.0001],
    "tau": [0.7, 1, 1.5, 2, 2.5, 3, 3.5, 4],
    "rho1": [0.1],
    "rho2": [0.05],
}
if RUN_CV:
    cv_dc = cu.tune_dc_admm(X_input, PARAM_GRID, n_folds=3, seed=42, max_dc_iter=10, max_admm_iter=15, n_jobs=N_JOBS)
    display(cv_dc.head(10))

In [ ]:
PARAMS = dict(lambda1=0.1, lambda2=0.0001, tau=3.5)
cluster_W, clusters, W_sub, labels = cu.optimize_dc_admm(
    X_input, **PARAMS, rho1=0.1, rho2=0.05, max_dc_iter=10, max_admm_iter=15, thres_value=0.01)

keys = list(X_sub)
pd.DataFrame({
    "cluster": range(1, len(clusters) + 1),
    "n_subpopulations": [len(c) for c in clusters],
    "n_cells": [sum(len(X_input[i]) for i in c) for c in clusters],
    "subpopulations": [sorted(keys[i] for i in c) for c in clusters],
})

## Reconstruction error (Table)

In [ ]:
table = pd.DataFrame({
    "Modeling scenario": ["Pooled population", "Per-perturbation", "DAG-DC-ADMM"],
    "Fitting granularity": [f"1 model ({sum(len(X) for X in X_input)} cells)", "9 models", f"{len(clusters)} models"],
    "Validation recon. error": [min(cv_pop.values()), cv_pert,
                                cv_dc["recon_mean"].iloc[0] if RUN_CV else np.nan],
})
table.round(2)

## Adjacency matrices (Figure)

In [ ]:
cu.plot_adjacency(
    [W_pop.T, cluster_W[1].T, cluster_W[3].T], cu.FEATURES,
    ["Adjacency Matrix of Population", "Adjacency Matrix of Cluster 2", "Adjacency Matrix of Cluster 4"],
    thr=0.04, out_path=f"{FIGURES}/case_study_adjacency.png")